# 02 - Preparação dos dados

Foram definidos os alvos Employability_Score (regressão) e Placement_Status (classificação), mantendo a seleção de atributos documentada na análise inicial. Os conjuntos processados preservam os valores originais; imputação e codificação são ajustadas dentro de pipelines durante o treinamento, evitando utilizar estatísticas do conjunto de teste.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

ROOT=Path('..'); RAW=ROOT/'data/raw/student_career_success_dataset.csv'; OUT=ROOT/'data/processed'
OUT.mkdir(parents=True,exist_ok=True)
df=pd.read_csv(RAW)
print('Dimensões',df.shape,'ausentes',df.isna().sum().sum())

Dimensões (50000, 29) ausentes 0


## 1. Targets e features mantidos da atividade anterior

In [2]:
reg_target='Employability_Score'; cls_target='Placement_Status'
reg_exclude=['Student_ID','Employability_Score','Placement_Status','Company_Tier','Career_Field','Placement_Mode','Starting_Salary_USD']
cls_exclude=['Student_ID','Placement_Status','Company_Tier','Career_Field','Placement_Mode','Starting_Salary_USD']
reg_features=[x for x in df.columns if x not in reg_exclude]
cls_features=[x for x in df.columns if x not in cls_exclude]
print('Regressão:',len(reg_features),reg_features)
print('Classificação:',len(cls_features),cls_features)

Regressão: 22 ['Age', 'Gender', 'University_Year', 'Major', 'Attendance_Percentage', 'Study_Hours_Per_Week', 'CGPA', 'Academic_Performance', 'Programming_Skill', 'Projects_Completed', 'Certifications', 'Hackathons', 'GitHub_Profile', 'Internships', 'Leadership_Experience', 'LinkedIn_Profile', 'Resume_Score', 'Communication_Skills', 'Teamwork', 'Problem_Solving', 'English_Proficiency', 'Interview_Score']
Classificação: 23 ['Age', 'Gender', 'University_Year', 'Major', 'Attendance_Percentage', 'Study_Hours_Per_Week', 'CGPA', 'Academic_Performance', 'Programming_Skill', 'Projects_Completed', 'Certifications', 'Hackathons', 'GitHub_Profile', 'Internships', 'Leadership_Experience', 'LinkedIn_Profile', 'Resume_Score', 'Communication_Skills', 'Teamwork', 'Problem_Solving', 'English_Proficiency', 'Interview_Score', 'Employability_Score']


## 2. Exportação sem ajustes globais

Os dados de saída preservam a escala e os valores categóricos originais. Isso evita estimar médias, desvios padrão e categorias sobre observações reservadas ao teste.

In [3]:
df.to_csv(OUT/'dataset_clean.csv',index=False)
df[reg_features+[reg_target]].to_csv(OUT/'dataset_regression.csv',index=False)
df[cls_features+[cls_target]].to_csv(OUT/'dataset_classification.csv',index=False)
print('Arquivos exportados.')

Arquivos exportados.


## 3. Divisão reprodutível por tarefa (80/20)

In [4]:
def export_split(target,filename,stratify=False):
    i=np.arange(len(df))
    train,test=train_test_split(i,test_size=0.2,random_state=42,stratify=df[target] if stratify else None)
    assignment=pd.DataFrame({'row_index':np.r_[train,test], 'split':['train']*len(train)+['test']*len(test)})
    assignment.sort_values('row_index').to_csv(OUT/filename,index=False)
    return train,test
reg_train,reg_test=export_split(reg_target,'split_regression.csv')
cls_train,cls_test=export_split(cls_target,'split_classification.csv',stratify=True)
print('Regressão',len(reg_train),len(reg_test),'| classificação',len(cls_train),len(cls_test))

Regressão 40000 10000 | classificação 40000 10000


## 4. Pré-processamento seguro para cada modelo

In [5]:
def build_preprocessor(X,scale_numeric=False):
    numerical=X.select_dtypes(include='number').columns.tolist()
    categorical=X.select_dtypes(exclude='number').columns.tolist()
    numeric_steps=[('imputer',SimpleImputer(strategy='median'))]
    if scale_numeric: numeric_steps.append(('scaler',StandardScaler()))
    return ColumnTransformer([
        ('numeric',Pipeline(numeric_steps),numerical),
        ('categorical',Pipeline([
            ('imputer',SimpleImputer(strategy='most_frequent')),
            ('encoder',OneHotEncoder(handle_unknown='ignore'))
        ]),categorical)
    ])
# EXEMPLO: nunca ajustar o transformador na totalidade dos dados.
X=df[reg_features]
processor=build_preprocessor(X,scale_numeric=False)
X_train_transformed=processor.fit_transform(X.iloc[reg_train])
X_test_transformed=processor.transform(X.iloc[reg_test])
print('Treino/teste transformados:',X_train_transformed.shape,X_test_transformed.shape)

Treino/teste transformados: (40000, 42) (10000, 42)


## Considerações sobre o pré-processamento

O particionamento treino/teste é reproduzível e compartilhado pelos modelos de cada tarefa. A normalização numérica não é indispensável às árvores utilizadas. O ajuste do pré-processador ocorre apenas no conjunto de treinamento e, durante a busca de hiperparâmetros, em cada divisão de validação cruzada.